# QuantumCircuit.jl Phase 2 Analysis Recipes

**Audience**
- Julia users who already have `SpectrumResult` or `SweepResult` objects and want to interpret them cleanly.

**Prerequisites**
- Basic Julia syntax.
- Familiarity with `spectrum`, `simulate_sweep`, and the Phase 2 subsystem types.

**What you will learn**
- How to read `SweepSeries` values returned by `transition_curve` and `anharmonicity_curve`.
- How to use `sweep_summary` for compact, table-friendly summaries.
- How to ask focused questions such as the minimum gap or a different transition.
- How to apply the same analysis helpers to subsystem sweeps and coupling sweeps.


## Outline

1. Activate the local package environment.
2. Create a tunable flux sweep result.
3. Read a `SweepSeries` and a `SweepSummary`.
4. Ask a more specific question with `minimum_gap` and a custom transition.
5. Reuse the same analysis helpers on a coupling sweep.
6. Review pitfalls and try a short exercise.


In [2]:
using Pkg

function find_repo_root(start::AbstractString)
    candidates = [
        normpath(start),
        normpath(joinpath(start, "..")),
        normpath(joinpath(start, "..", "..")),
    ]

    for candidate in unique(candidates)
        project_toml = joinpath(candidate, "Project.toml")
        if isfile(project_toml)
            content = read(project_toml, String)
            occursin("QuantumCircuit", content) && return candidate
        end
    end

    error("Could not find the QuantumCircuit.jl project root. Start Jupyter from the repository or open the notebook from inside it.")
end

project_root = find_repo_root(pwd())
Pkg.activate(project_root)
Pkg.instantiate()

using QuantumCircuit


  Activating project at `~/Research/20_Projects/QuantumCircuit.jl`


## Step 1 - Create a sweep result to analyze

Start from a simple tunable transmon sweep. The point of this notebook is not the model construction itself, but what you can do once you already have a `SweepResult`.


In [3]:
tq = TunableTransmon(:tq; EJmax = 20.0, EC = 0.25, flux = 0.0, asymmetry = 0.0, ncut = 6)
sys = CompositeSystem(tq)
sweep = SweepSpec(:tq, :flux, [0.0, 0.15, 0.30]; levels = 4)
result = simulate_sweep(sys, sweep)

(
    target = result.spec.target,
    parameter = result.spec.parameter,
    values = result.values,
    spectra_count = length(result.spectra),
)


(target = SubsystemSweepTarget(:tq), parameter = :flux, values = [0.0, 0.15, 0.3], spectra_count = 3)

## Step 2 - Read a `SweepSeries`

`transition_curve` and `anharmonicity_curve` both return a `SweepSeries`. This is a lightweight container designed to be easy to inspect or pass to a plotting layer later.


In [4]:
ω01_curve = transition_curve(result)
alpha_curve = anharmonicity_curve(result)

(
    transition_label = ω01_curve.label,
    transition_values = ω01_curve.values,
    transition_data = ω01_curve.data,
    anharmonicity_label = alpha_curve.label,
    anharmonicity_data = alpha_curve.data,
)


(transition_label = :transition_01, transition_values = [0.0, 0.15, 0.3], transition_data = [6.074555320336759, 5.719946479453121, 4.598856575698948], anharmonicity_label = :anharmonicity, anharmonicity_data = [-0.24999999999999645, -0.24999999999999822, -0.24999999999999822])

## Step 3 - Build a compact summary

Use `sweep_summary` when you want the default `0 -> 1` transition and anharmonicity packaged together in one object.


In [5]:
summary = sweep_summary(result)

(
    values = summary.values,
    transition_01 = summary.transition_01,
    anharmonicity = summary.anharmonicity,
)


(values = [0.0, 0.15, 0.3], transition_01 = [6.074555320336759, 5.719946479453121, 4.598856575698948], anharmonicity = [-0.24999999999999645, -0.24999999999999822, -0.24999999999999822])

## Step 4 - Ask a more specific question

You can also request a different transition or ask for the minimum gap along the sweep.


In [6]:
transition_02 = transition_curve(result; transition = (1, 3))
gap = minimum_gap(result)

(
    transition_02_label = transition_02.label,
    transition_02_data = transition_02.data,
    minimum_gap = gap,
)


(transition_02_label = :transition_02, transition_02_data = [11.899110640673522, 11.189892958906244, 8.947713151397897], minimum_gap = (gap = 4.598856575698948, sweep_value = 0.3, index = 3, level_pair = (1, 2)))

## Step 5 - Reuse the same helpers on a coupling sweep

The Analysis API does not care whether the sweep changed a subsystem parameter or a coupling parameter. It only reads the resulting `SweepResult`.


In [7]:
q1 = Transmon(:q1; EJ = 20.0, EC = 0.25, ncut = 5)
r1 = Resonator(:r1; ω = 6.8, dim = 4)

coupled_sys = CompositeSystem(
    q1,
    r1,
    CapacitiveCoupling(:q1, :r1; g = 0.02),
)

coupling_sweep = SweepSpec(:q1, :r1, :g, [0.02, 0.08, 0.14]; levels = 5)
coupling_result = simulate_sweep(coupled_sys, coupling_sweep)
coupling_curve = transition_curve(coupling_result)
coupling_summary = sweep_summary(coupling_result)

(
    coupling_values = coupling_summary.values,
    transition_01 = coupling_curve.data,
    first_spectrum = coupling_result.spectra[1].energies,
)


(coupling_values = [0.02, 0.08, 0.14], transition_01 = [6.074004352846019, 6.065837899489844, 6.0484750163850265], first_spectrum = [0.0, 6.074004352846019, 6.800550967490739, 11.898290795844863, 12.874273352821513])

## Pitfalls and Extensions

**Common pitfall**
- `sweep_summary` assumes each spectrum has enough levels to compute both the `0 -> 1` transition and the anharmonicity.

**Useful extension**
- If you later add plotting, `SweepSeries.values` and `SweepSeries.data` are already shaped for a straightforward x-y plot.


In [8]:
try
    bad_summary = sweep_summary(simulate_sweep(sys, SweepSpec(:tq, :flux, [0.0, 0.15]; levels = 2)))
catch err
    sprint(showerror, err)
end


"ArgumentError: anharmonicity requires at least three energy levels."

## Exercise

Create a new sweep with a different flux grid and answer two questions:

1. How does the `0 -> 2` transition change compared with the `0 -> 1` transition?
2. Does the minimum gap still happen at the largest flux value in your new grid?

Use the next cell as a scaffold.


In [10]:
exercise_result = simulate_sweep(sys, SweepSpec(:tq, :flux, [0.0, 0.05, 0.10, 0.20, 0.30]; levels = 4))
exercise_transition_01 = transition_curve(exercise_result)
exercise_transition_02 = transition_curve(exercise_result; transition = (1, 3))
exercise_gap = minimum_gap(exercise_result)

(
    flux_values = exercise_result.values,
    transition_01 = exercise_transition_01.data,
    transition_02 = exercise_transition_02.data,
    minimum_gap = exercise_gap,
)


(flux_values = [0.0, 0.05, 0.1, 0.2, 0.3], transition_01 = [6.074555320336759, 6.035501859343095, 5.917840841964564, 5.438644810057831, 4.598856575698948], transition_02 = [11.899110640673522, 11.821003718686194, 11.58568168392913, 10.627289620115663, 8.947713151397897], minimum_gap = (gap = 4.598856575698948, sweep_value = 0.3, index = 5, level_pair = (1, 2)))